# 实验六：Transformer 与本地小语言模型

本实验目标是让你在自己的电脑上跑一个真正的开源语言模型，同时把 Transformer 的关键部件拆开看：tokenizer、chat template、因果注意力、next-token 概率、采样策略和自回归生成。

我们平时看到的大语言模型像是在“回答问题”，但从模型内部看，它做的是一件非常具体的事：给定已经出现的 token 序列，预测下一个 token 的概率分布。无论是聊天、写代码、翻译、摘要，最后都会被组织成这样的 next-token prediction 问题。本实验会沿着这条主线展开：

1. 文本先被 tokenizer 切成 token id。
2. 对话消息会被 chat template 包装成模型熟悉的格式。
3. Transformer 解码器用 causal self-attention 处理上下文。
4. 模型输出 logits，softmax 后变成下一个 token 的概率。
5. 解码器按某种策略选择下一个 token，再把它接回上下文继续生成。

本 notebook 默认使用 `HuggingFaceTB/SmolLM2-135M-Instruct`。它是 Apache-2.0 许可证的小型指令模型，约 0.1B 参数，第一次运行需要联网下载，之后会缓存在本机。它的中文能力有限，但体积小、下载快、适合课堂上理解原理。想要中文输出更强，可以在课后把 `MODEL_ID` 改成更大的中文模型再比较。

请使用 `pt` kernel。所有需要你填写的代码都放在 `START CODE HERE` 和 `END CODE HERE` 标记之间；发布学生版时会自动去掉这些区域里的教师答案。

本实验的重点不是把小模型调到“最好用”，而是理解它为什么能生成、为什么会犯错、哪些参数会改变输出。小模型容易胡说，尤其是数学、事实和中文长文本；请把它当作“可拆开的语言模型玩具”，不要把输出当作可靠答案。


## 0. 本地环境准备

第一次运行可能会下载约数百 MB 的模型文件。下载完成后，Hugging Face 会把模型缓存在本机，后续运行通常不需要重复下载。CPU 可以运行，但生成速度会慢一些；有 Apple Silicon 的同学会自动尝试 MPS；有 NVIDIA GPU 的同学会使用 CUDA。

本实验主要用到两个库：

- `torch`：负责张量计算、模型前向传播和采样。
- `transformers`：负责从 Hugging Face 加载 tokenizer、模型权重和模型配置。

运行语言模型时，最容易遇到的本地问题通常有三类：

1. 依赖缺失：看到 `ModuleNotFoundError` 时，先确认当前 notebook 使用的是 `pt` kernel，再安装缺失包。
2. 模型下载慢：下载 Hugging Face 模型较慢时，可以在启动 Jupyter 前设置镜像环境变量，例如 `export HF_ENDPOINT=https://hf-mirror.com`。
3. 内存不足：如果电脑内存较小，尽量关闭其他占内存程序；生成时把 `max_new_tokens` 调小也会更稳。

如果缺少依赖，先取消下一格里 `%pip install` 那一行的注释并运行。安装后建议重启 kernel，再从头运行 notebook。


In [25]:
# 如缺少依赖，取消下一行注释并运行一次；安装后重启 kernel。
# %pip install -U torch transformers accelerate safetensors

import importlib.util

missing = [pkg for pkg in ["torch", "transformers"] if importlib.util.find_spec(pkg) is None]
if missing:
    print("缺少依赖：", missing)
    print("请先运行：%pip install -U torch transformers accelerate safetensors")
else:
    print("依赖检查通过。")


依赖检查通过。


In [26]:
import math
import random
import textwrap
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

MODEL_ID = "HuggingFaceTB/SmolLM2-135M-Instruct"

def pick_device():
    """选择当前机器上最合适的推理设备。"""
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = pick_device()
# CPU/MPS 用 float32 最稳；CUDA 可以用 float16 节省显存。
dtype = torch.float16 if device.type == "cuda" else torch.float32

print("device:", device)
print("dtype:", dtype)


device: mps
dtype: torch.float32


In [27]:
# 第一次运行会从 Hugging Face 下载模型；之后会从本机缓存读取。
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=dtype)
model.to(device)
model.eval()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

n_params = sum(p.numel() for p in model.parameters())
print(f"Loaded {MODEL_ID}")
print(f"parameters: {n_params / 1e6:.1f}M")
print("vocab size:", len(tokenizer))
print("context length:", getattr(model.config, "max_position_embeddings", "unknown"))


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Loaded HuggingFaceTB/SmolLM2-135M-Instruct
parameters: 134.5M
vocab size: 49152
context length: 8192


## 1. Tokenizer：文字如何变成 token

语言模型并不是直接读取“汉字”或“英文单词”，而是读取 token id。模型的输入本质上是一个整数序列，例如：

```text
"Transformer is useful" -> [10680, 336, 5632]
```

这些整数来自 tokenizer 的词表。词表就像一张“字符串片段到整数 id 的表”：某个 id 可能表示一个完整英文单词，也可能表示一个词根、一个标点、一个空格加单词、一个中文字符，甚至只是 UTF-8 字节片段。现代大模型常用 BPE、WordPiece、Unigram 或 byte-level BPE 一类方法，把常见片段合成较长 token，把少见片段拆成更小 token。

理解 tokenizer 很重要，因为模型真正看到的不是你屏幕上的自然语言，而是 token 序列。下面几个现象在大模型实验中非常常见：

- 英文里的空格经常和后面的词一起组成 token，所以 `' cat'` 和 `'cat'` 可能是不同 token。
- 一个中文词可能被拆成多个 token，小模型的中文输出不稳定，常常和 tokenizer、训练数据都有关系。
- 代码里的缩进、换行、括号也都会被 token 化，因此代码补全对格式很敏感。
- prompt 里多一个标点、空格或换行，都可能改变后续 token 序列，从而改变模型输出。

### 练习 1：写一个 token 观察器

请实现 `token_table`，把一段文本拆成 token，并返回前若干个 token 的 id、内部字符串和可读文本片段。这个函数后面会用于观察 prompt 为什么会影响模型输出。

实现时要分清三种表示：

- `id`：模型真正使用的整数。
- `token`：tokenizer 内部使用的字符串，可能带特殊空格符号或字节符号。
- `text`：把单个 token decode 回来以后，人类较容易阅读的片段。

完成这个练习后，你应该能回答：同一句话中哪些片段是一个 token，哪些片段被拆开；中文、英文和代码的切分方式有什么不同。

本练习只需要补 2 行：一行编码文本，一行把 id 转回 tokenizer 内部 token。表格整理代码已经给出。


In [28]:
def _format_token_rows(ids, tokens, max_rows):
    """把 token id 和内部 token 字符串整理成便于观察的表格。"""
    return [
        {
            "pos": pos,
            "id": int(token_id),
            "token": token,
            "text": tokenizer.decode([token_id]),
        }
        for pos, (token_id, token) in enumerate(zip(ids[:max_rows], tokens[:max_rows]))
    ]


def token_table(text, max_rows=20):
    """
    返回 list[dict]，每个 dict 描述一个 token：
    - pos: token 在序列中的位置
    - id: tokenizer 产生的整数 id
    - token: tokenizer 的内部 token 字符串
    - text: 单独 decode 这个 token 后得到的人类可读片段
    """
    ### START CODE HERE ###
    # TODO 1：把 text 编码成 token id 列表。
    # 提示：使用 tokenizer.encode(..., add_special_tokens=False)。

    # TODO 2：把 token id 列表转换成 tokenizer 内部 token 字符串。
    # 提示：使用 tokenizer.convert_ids_to_tokens(ids)。
    ### END CODE HERE ###

    return _format_token_rows(ids, tokens, max_rows)

sample_text = "A tiny Transformer can still be interesting. 小模型也能帮助我们理解原理。"
rows = token_table(sample_text, max_rows=24)
rows


[{'pos': 0, 'id': 49, 'token': 'A', 'text': 'A'},
 {'pos': 1, 'id': 5383, 'token': 'Ġtiny', 'text': ' tiny'},
 {'pos': 2, 'id': 3790, 'token': 'ĠTrans', 'text': ' Trans'},
 {'pos': 3, 'id': 20714, 'token': 'former', 'text': 'former'},
 {'pos': 4, 'id': 416, 'token': 'Ġcan', 'text': ' can'},
 {'pos': 5, 'id': 1361, 'token': 'Ġstill', 'text': ' still'},
 {'pos': 6, 'id': 325, 'token': 'Ġbe', 'text': ' be'},
 {'pos': 7, 'id': 3684, 'token': 'Ġinteresting', 'text': ' interesting'},
 {'pos': 8, 'id': 30, 'token': '.', 'text': '.'},
 {'pos': 9, 'id': 13223, 'token': 'Ġå', 'text': ' �'},
 {'pos': 10, 'id': 125, 'token': '°', 'text': '�'},
 {'pos': 11, 'id': 233, 'token': 'ı', 'text': '�'},
 {'pos': 12, 'id': 43828, 'token': 'æ¨', 'text': '�'},
 {'pos': 13, 'id': 111, 'token': '¡', 'text': '�'},
 {'pos': 14, 'id': 46157, 'token': 'åŀĭ', 'text': '型'},
 {'pos': 15, 'id': 27948, 'token': 'ä¹', 'text': '�'},
 {'pos': 16, 'id': 249, 'token': 'Ł', 'text': '�'},
 {'pos': 17, 'id': 47093, 'token': 'èĥ

In [29]:
# 测试练习 1
rows = token_table("Transformer tokens are not always words.", max_rows=10)
assert isinstance(rows, list), "token_table 应返回 list"
assert len(rows) > 0, "返回结果不能为空"
assert {"pos", "id", "token", "text"}.issubset(rows[0].keys()), "每行需要包含 pos/id/token/text"
assert all(isinstance(r["id"], int) for r in rows), "id 应该是 Python int"
assert rows[0]["pos"] == 0, "pos 应从 0 开始"
print("练习 1 通过。")


练习 1 通过。


## 2. Chat template：指令模型看到的并不是原始问题

基础语言模型只学习“给定前文预测下一个 token”。指令模型在这个基础上又经过了监督微调和偏好对齐，使它更擅长按照人类指令回答。为了让模型区分“系统设定”“用户问题”和“助手回答”，训练时通常会给对话加入固定格式，例如：

```text
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Explain attention.<|im_end|>
<|im_start|>assistant
```

这类格式就叫 chat template。不同模型的 template 可能不同：有的使用 `<|im_start|>`，有的使用 `[INST]...[/INST]`，有的使用其他特殊 token。如果格式不匹配，模型可能仍然能输出文字，但会更容易跑题、复读、忽略 system prompt，或者把用户问题当成普通续写文本。

在 Hugging Face 中，`tokenizer.apply_chat_template` 会根据模型 tokenizer 自带的配置，把 OpenAI 风格的 `messages` 列表转换成模型训练时熟悉的字符串或 token id。这里的 `messages` 一般包含三类角色：

- `system`：给模型的全局行为约束，例如“回答要简洁”。
- `user`：用户提出的问题或任务。
- `assistant`：历史回答；多轮对话时会把之前的回答也放进去。

`add_generation_prompt=True` 尤其关键。它会把模板补到“assistant 应该开始回答”的位置。如果忘了加，模型可能不知道现在轮到谁说话。

### 练习 2：把 messages 变成模型输入

请实现 `build_chat_inputs`。它接收 OpenAI 风格的 `messages` 列表，返回可以直接喂给模型的张量字典。

你需要完成两步转换：

1. `messages -> input_text`：用 chat template 生成模型真正看到的对话字符串。
2. `input_text -> tensors`：用 tokenizer 把字符串转成 `input_ids` 和 `attention_mask`，并移动到当前设备。

完成这个练习后，你可以打印 decode 后的 `input_ids`，观察用户写下的问题在进入模型之前被包装成了什么样子。

本练习只需要补 2 行：先套 chat template，再调用 tokenizer 生成张量。移动到设备的代码已经给出。


In [30]:
def build_chat_inputs(messages, tokenizer, device):
    """
    messages 示例：
    [
        {"role": "system", "content": "You are concise."},
        {"role": "user", "content": "Explain attention."},
    ]

    返回值应至少包含 input_ids；如果 tokenizer 产生 attention_mask，也一起返回。
    所有张量都要移动到 device 上。
    """
    ### START CODE HERE ###
    # TODO 1：用 chat template 把 messages 转成模型真正看到的字符串。
    # 提示：tokenize=False，add_generation_prompt=True。

    # TODO 2：把字符串 tokenizer 成 PyTorch 张量。
    # 提示：return_tensors="pt"。
    ### END CODE HERE ###

    return {key: value.to(device) for key, value in encoded.items()}

messages = [
    {"role": "system", "content": "You answer in one short sentence."},
    {"role": "user", "content": "What is a Transformer?"},
]
inputs = build_chat_inputs(messages, tokenizer, device)
print(inputs.keys())
print(inputs["input_ids"].shape)
print(tokenizer.decode(inputs["input_ids"][0][:80]))


dict_keys(['input_ids', 'attention_mask'])
torch.Size([1, 27])
<|im_start|>system
You answer in one short sentence.<|im_end|>
<|im_start|>user
What is a Transformer?<|im_end|>
<|im_start|>assistant



In [31]:
# 测试练习 2
messages = [{"role": "user", "content": "Say hello in five words."}]
inputs = build_chat_inputs(messages, tokenizer, device)
assert isinstance(inputs, dict), "build_chat_inputs 应返回 dict"
assert "input_ids" in inputs, "返回值必须包含 input_ids"
assert inputs["input_ids"].ndim == 2, "input_ids 形状应为 [batch, seq_len]"
assert inputs["input_ids"].device.type == device.type, "张量需要移动到当前 device"
print("练习 2 通过。")


练习 2 通过。


## 3. 先让模型说一句话

这一节不用填代码，只确认本地模型能完成一次生成。CPU 上第一次生成可能较慢。

请注意观察两件事：

1. 输入给 `model.generate` 的不是普通字符串，而是上一节得到的 `input_ids` 和 `attention_mask`。
2. `generate` 返回的是完整序列，也就是“原始 prompt token + 新生成 token”。如果只想看助手新说的话，需要把 prompt 部分切掉，只 decode 新生成的 token。

这个 API 很方便，但它隐藏了很多关键步骤：每一步取最后位置 logits、选择一个 token、拼回序列、继续前向传播。后面的练习会把这些步骤拆开实现。


In [32]:
@torch.no_grad()
def quick_chat(user_prompt, max_new_tokens=60, temperature=0.7):
    messages = [
        {"role": "system", "content": "You are a helpful teaching assistant. Keep answers short."},
        {"role": "user", "content": user_prompt},
    ]
    inputs = build_chat_inputs(messages, tokenizer, device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=temperature > 0,
        temperature=max(temperature, 1e-6),
        pad_token_id=tokenizer.eos_token_id,
    )
    new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

print(quick_chat("Explain self-attention to a beginner in two sentences."))


Self-attention is a technique used in AI to learn to recognize and focus on specific aspects of a text or image. It involves continually scanning the surrounding area of the text or image, making adjustments to focus on the most important information. It's a way to refine your understanding of the content and to


## 4. 从零实现因果自注意力

Transformer 的核心是 self-attention。它让序列中每个位置都可以根据其他位置的信息更新自己的表示。以一句话为例，模型在处理某个 token 时，会计算它应该“关注”前面哪些 token：是主语、谓语、括号里的解释，还是代码里的变量名。

Self-attention 通常会把每个位置的隐藏向量线性变换成三组向量：

- Query (`Q`)：当前位置在“寻找什么信息”。
- Key (`K`)：每个位置提供的“索引”或“匹配特征”。
- Value (`V`)：每个位置真正贡献给输出的内容。

注意力分数来自 `QK^T`。如果某个 query 和某个 key 的点积大，就表示当前位置更应该关注那个位置。除以 `sqrt(d_k)` 是为了避免维度较大时点积数值过大，导致 softmax 过早饱和，梯度变得不稳定。

对于 decoder-only 语言模型，还必须使用 causal mask。原因是训练目标是预测下一个 token：位置 `i` 只能利用 `0..i` 的信息，不能偷看未来的 `i+1..n`。如果训练时能偷看未来，模型会学到一种作弊规则；到了真实生成时未来 token 并不存在，训练和推理就不一致了。

因果 mask 的形状可以理解为一张可见性表：

```text
位置 0: 只能看 0
位置 1: 可以看 0, 1
位置 2: 可以看 0, 1, 2
位置 3: 可以看 0, 1, 2, 3
```

### 练习 3：实现 causal mask 和 scaled dot-product attention

这里不调用 PyTorch 的 `scaled_dot_product_attention`，而是手写核心公式：

$$
\mathrm{Attention}(Q,K,V)=\mathrm{softmax}\left(\frac{QK^T}{\sqrt{d_k}} + \mathrm{mask}\right)V
$$

实现时请特别注意张量形状：

- `q, k, v` 的形状是 `[batch, heads, seq_len, head_dim]`。
- `q @ k.transpose(-2, -1)` 的形状是 `[batch, heads, seq_len, seq_len]`。
- mask 的形状是 `[seq_len, seq_len]`，可以自动 broadcast 到 batch 和 head 维度。
- 被屏蔽的位置要填成 `-inf`，这样 softmax 后对应概率就是 0。

完成这个练习后，你会看到未来位置的注意力权重确实为 0，并且每个位置的注意力权重仍然会归一化为 1。

本练习分成两个很短的小填空：mask 只需 1 行；attention 只需补分数、mask 和 softmax 这 3 个关键步骤。


In [33]:
def make_causal_mask(seq_len, device=None):
    """
    返回形状为 [seq_len, seq_len] 的 bool 矩阵。
    mask[i, j] == True 表示第 i 个位置允许看第 j 个位置。
    因果语言模型中，位置 i 只能看 0..i，不能看 i+1..seq_len-1。
    """
    ### START CODE HERE ###
    # TODO：创建一个 bool 下三角矩阵。
    # 提示：torch.ones(..., dtype=torch.bool)，再用 torch.tril(...)。
    ### END CODE HERE ###
    return mask


def scaled_dot_product_attention(q, k, v, allowed_mask=None):
    """
    q, k, v 形状均为 [batch, heads, seq_len, head_dim]。
    allowed_mask 形状为 [seq_len, seq_len]，True 表示允许看。

    返回：
    - context: [batch, heads, seq_len, head_dim]
    - weights: [batch, heads, seq_len, seq_len]
    """
    head_dim = q.shape[-1]

    ### START CODE HERE ###
    # TODO 1：计算缩放点积注意力分数。
    # 提示：q @ k.transpose(-2, -1)，再除以 math.sqrt(head_dim)。

    # TODO 2：如果给了 allowed_mask，把未来位置填成 -inf。
    # 提示：False 的位置要屏蔽，可以使用 masked_fill(~allowed_mask, float("-inf"))。

    # TODO 3：对最后一维做 softmax，得到注意力权重。
    ### END CODE HERE ###

    context = weights @ v
    return context, weights

# 一个小玩具输入：batch=1, heads=2, seq_len=4, head_dim=3
torch.manual_seed(SEED)
q = torch.randn(1, 2, 4, 3)
k = torch.randn(1, 2, 4, 3)
v = torch.randn(1, 2, 4, 3)
mask = make_causal_mask(seq_len=4, device=q.device)
context, weights = scaled_dot_product_attention(q, k, v, mask)
print("mask:\n", mask.int())
print("context shape:", context.shape)
print("weights shape:", weights.shape)


mask:
 tensor([[1, 0, 0, 0],
        [1, 1, 0, 0],
        [1, 1, 1, 0],
        [1, 1, 1, 1]], dtype=torch.int32)
context shape: torch.Size([1, 2, 4, 3])
weights shape: torch.Size([1, 2, 4, 4])


In [34]:
# 测试练习 3
mask = make_causal_mask(5)
assert mask.dtype == torch.bool, "mask 应为 bool tensor"
assert mask.shape == (5, 5), "mask 形状应为 [seq_len, seq_len]"
assert mask[4, 0] and mask[4, 4], "当前位置应能看见过去和自己"
assert not mask[0, 1] and not mask[1, 4], "不应看见未来"

torch.manual_seed(0)
q = torch.randn(2, 3, 5, 4)
k = torch.randn(2, 3, 5, 4)
v = torch.randn(2, 3, 5, 4)
context, weights = scaled_dot_product_attention(q, k, v, mask)
assert context.shape == q.shape, "context 形状应与 q 相同"
assert weights.shape == (2, 3, 5, 5), "weights 形状错误"
future_weights = torch.triu(weights, diagonal=1)
assert torch.allclose(future_weights, torch.zeros_like(future_weights), atol=1e-6), "未来位置的注意力权重应为 0"
assert torch.allclose(weights.sum(dim=-1), torch.ones_like(weights.sum(dim=-1)), atol=1e-5), "每行注意力权重应和为 1"
print("练习 3 通过。")


练习 3 通过。


## 5. 看看真实模型的结构

前面手写的是 attention 的最小核心。真实语言模型会把它堆叠成多层 Transformer block，每层通常包含：

1. 归一化层，例如 RMSNorm 或 LayerNorm。
2. 多头自注意力，用多个 head 从不同角度读取上下文。
3. 前馈网络，通常是一个升维再降维的 MLP。
4. 残差连接，让深层网络更容易训练。

模型配置里的几个字段值得关注：

- `hidden_size`：每个 token 的隐藏向量维度。
- `num_hidden_layers`：Transformer block 的层数。
- `num_attention_heads`：注意力头数。
- `num_key_value_heads`：如果小于注意力头数，说明模型使用了 grouped-query attention 之类的节省 KV cache 的结构。
- `intermediate_size`：前馈网络中间层维度。
- `max_position_embeddings`：模型理论上支持的最大上下文长度。

注意力的显存/内存开销与序列长度近似平方相关，因为每层每个 head 都会产生 `[seq_len, seq_len]` 的注意力矩阵。即使模型参数不大，长上下文也可能让推理变慢或占用大量内存。

下面只读取配置，不训练模型。请把这些配置和前面手写 attention 的张量形状对应起来看。


In [35]:
config_items = {
    "model_type": getattr(model.config, "model_type", None),
    "hidden_size": getattr(model.config, "hidden_size", None),
    "num_hidden_layers": getattr(model.config, "num_hidden_layers", None),
    "num_attention_heads": getattr(model.config, "num_attention_heads", None),
    "num_key_value_heads": getattr(model.config, "num_key_value_heads", None),
    "intermediate_size": getattr(model.config, "intermediate_size", None),
    "max_position_embeddings": getattr(model.config, "max_position_embeddings", None),
}
config_items


{'model_type': 'llama',
 'hidden_size': 576,
 'num_hidden_layers': 30,
 'num_attention_heads': 9,
 'num_key_value_heads': 3,
 'intermediate_size': 1536,
 'max_position_embeddings': 8192}

In [36]:
# 估算一个 batch 的注意力权重张量大小。
# 注意力权重形状近似为 [batch, heads, seq_len, seq_len]，所以长上下文会二次增长。

def estimate_attention_memory_mb(batch, heads, seq_len, bytes_per_number=4):
    n_numbers = batch * heads * seq_len * seq_len
    return n_numbers * bytes_per_number / (1024 ** 2)

heads = getattr(model.config, "num_attention_heads", 1)
for seq_len in [128, 512, 1024, 2048]:
    mb = estimate_attention_memory_mb(batch=1, heads=heads, seq_len=seq_len, bytes_per_number=4)
    print(f"seq_len={seq_len:4d}: attention weights about {mb:8.1f} MB in float32")


seq_len= 128: attention weights about      0.6 MB in float32
seq_len= 512: attention weights about      9.0 MB in float32
seq_len=1024: attention weights about     36.0 MB in float32
seq_len=2048: attention weights about    144.0 MB in float32


## 6. Next-token probability：模型下一步想说什么

语言模型每一步都输出一个长度等于词表大小的 logits 向量。logits 还不是概率，它们是未归一化的分数，可以是任意实数。经过 softmax 后，logits 会变成词表上每个 token 的概率：

$$
p_i = \frac{\exp(z_i)}{\sum_j \exp(z_j)}
$$

其中 `z_i` 是第 `i` 个 token 的 logit，`p_i` 是它成为下一个 token 的概率。模型生成文本时，真正发生的是：它在每一步都给整个词表打分，然后根据某种策略选择一个 token。

这个视角能解释很多现象：

- 模型不是先想好一整段话，而是一步一步往后续写。
- 一个回答看起来很确定，不代表模型内部概率真的集中；很多候选 token 可能概率接近。
- prompt 改一个词，最后位置的 hidden state 会变，下一步概率分布也会变。
- 小模型经常把开头写得像样，但越写越偏，因为每一步的小误差都会进入后续上下文。

`top_n` 候选 token 是观察模型行为的好工具。它不会告诉我们“正确答案”，但能展示模型在当前上下文下最倾向于接什么。

### 练习 4：实现候选 token 查看器

请实现 `next_token_candidates`，输入用户 prompt，输出模型认为最可能的若干个下一个 token。

实现时要抓住一个关键位置：`outputs.logits[0, -1]`。这里的 `-1` 表示当前上下文最后一个 token 的输出位置，也就是模型对“下一个 token”的预测。对这个向量做 softmax，再取 top-k，就能看到候选 token 和概率。

完成这个练习后，建议你多换几种 prompt，观察英文、中文、代码开头的候选 token 是否符合直觉。

本练习只需要补 3 行：取最后位置 logits、softmax、top-k。候选表格整理代码已经给出。


In [37]:
def _format_top_candidates(top_ids, top_probs, tokenizer):
    """把 top-k 结果整理成人类可读的候选 token 表。"""
    return [
        {
            "rank": rank,
            "token_id": int(token_id),
            "token": tokenizer.decode([int(token_id)]),
            "prob": float(prob),
        }
        for rank, (token_id, prob) in enumerate(zip(top_ids.tolist(), top_probs.tolist()), start=1)
    ]


@torch.no_grad()
def next_token_candidates(user_prompt, top_n=10):
    """
    返回 top_n 个候选 token。每项包含：
    - rank: 从 1 开始的排名
    - token_id: token id
    - token: decode 后的人类可读片段
    - prob: softmax 概率
    """
    messages = [
        {"role": "system", "content": "You are concise."},
        {"role": "user", "content": user_prompt},
    ]
    inputs = build_chat_inputs(messages, tokenizer, device)
    outputs = model(**inputs)

    ### START CODE HERE ###
    # TODO 1：取最后一个位置的 logits。
    # 提示：outputs.logits 的形状是 [batch, seq_len, vocab_size]。

    # TODO 2：把 logits 变成概率。

    # TODO 3：取概率最高的 top_n 个 token。
    # 提示：torch.topk 返回 values 和 indices。
    ### END CODE HERE ###

    return _format_top_candidates(top_ids, top_probs, tokenizer)

candidates = next_token_candidates("Complete this sentence: Deep learning is", top_n=8)
candidates


[{'rank': 1, 'token_id': 29602, 'token': 'Deep', 'prob': 0.6375501155853271},
 {'rank': 2, 'token_id': 504, 'token': 'The', 'prob': 0.06330325454473495},
 {'rank': 3, 'token_id': 13701, 'token': 'AI', 'prob': 0.01098452229052782},
 {'rank': 4, 'token_id': 18, 'token': '"', 'prob': 0.010747557505965233},
 {'rank': 5, 'token_id': 788, 'token': 'In', 'prob': 0.009713869541883469},
 {'rank': 6, 'token_id': 1882, 'token': 'We', 'prob': 0.007955070585012436},
 {'rank': 7, 'token_id': 3825, 'token': 'With', 'prob': 0.007640341762453318},
 {'rank': 8, 'token_id': 49, 'token': 'A', 'prob': 0.006919966544955969}]

In [38]:
# 测试练习 4
candidates = next_token_candidates("Complete this sentence: The capital of France is", top_n=5)
assert isinstance(candidates, list) and len(candidates) == 5, "应返回 top_n 个候选项"
assert {"rank", "token_id", "token", "prob"}.issubset(candidates[0].keys()), "候选项字段不完整"
assert candidates[0]["rank"] == 1, "rank 应从 1 开始"
assert all(candidates[i]["prob"] >= candidates[i + 1]["prob"] for i in range(len(candidates) - 1)), "概率应降序排列"
assert 0 <= candidates[0]["prob"] <= 1, "prob 应是概率"
print("练习 4 通过。")


练习 4 通过。


### 小实验：模型猜词游戏

换几个 prompt，观察“下一个 token”如何变化。你可以尝试：

- `The best way to learn PyTorch is`
- `In a neural network, attention means`
- `请用一句话解释 Transformer：`
- `def fibonacci(n):`

问题：为什么有些 token 看起来只是一个空格或半个词？为什么中文 prompt 的候选可能不稳定？


In [39]:
for prompt in [
    "The best way to learn PyTorch is",
    "In a neural network, attention means",
    "请用一句话解释 Transformer：",
    "def fibonacci(n):",
]:
    print("\nPROMPT:", prompt)
    for row in next_token_candidates(prompt, top_n=5):
        print(f"  {row['rank']:>2}. {row['token']!r:<12} p={row['prob']:.4f}")



PROMPT: The best way to learn PyTorch is
   1. 'The'        p=0.2764
   2. 'You'        p=0.0716
   3. 'Py'         p=0.0514
   4. 'To'         p=0.0441
   5. 'Here'       p=0.0440

PROMPT: In a neural network, attention means
   1. 'In'         p=0.3782
   2. 'Attention'  p=0.2269
   3. 'The'        p=0.0716
   4. '"'          p=0.0253
   5. 'You'        p=0.0210

PROMPT: 请用一句话解释 Transformer：
   1. 'Trans'      p=0.3742
   2. 'The'        p=0.2338
   3. 'A'          p=0.0704
   4. 'Transform'  p=0.0651
   5. 'In'         p=0.0130

PROMPT: def fibonacci(n):
   1. '```'        p=0.2091
   2. 'Here'       p=0.1067
   3. 'The'        p=0.1033
   4. 'A'          p=0.1016
   5. 'def'        p=0.0618


## 7. 采样策略：temperature、top-k 与 top-p

如果总是选择概率最大的 token，这叫 greedy decoding。它输出稳定、可复现，但也容易变得重复、保守，甚至陷入循环。开放式写作、头脑风暴、故事生成等任务通常会使用采样，让模型从概率分布中随机抽取 token。

采样不是“随便乱选”，而是在模型给出的概率分布上做受控随机。常见控制参数有三个：

- `temperature` 控制分布尖锐程度。把 logits 除以小于 1 的 temperature，会让大 logit 更突出，输出更保守；除以大于 1 的 temperature，会让分布更平，输出更多样但也更容易跑题。
- `top_k` 只保留概率最高的 k 个 token，其他 token 直接屏蔽。它能防止模型抽到非常离谱的低概率 token。
- `top_p` 也叫 nucleus sampling。它按概率从大到小累加，只保留累计概率达到 p 的最小集合。和固定 `top_k` 相比，`top_p` 会根据当前分布自动调整候选集合大小：模型很确定时保留很少 token，模型不确定时保留更多 token。

屏蔽 token 时，我们通常把对应 logits 设成 `-inf`。这样 softmax 后它们的概率就是 0。注意：top-k 和 top-p 操作应该作用在 logits 上，然后再 softmax；如果先 softmax 再过滤，也能做，但数值和实现细节更容易出错。

### 练习 5：实现 logits 过滤器

请实现 `filter_logits`。它返回过滤后的 logits，被屏蔽的 token 位置应为 `-inf`。

你需要按顺序完成：复制 logits、应用 temperature、应用 top-k、应用 top-p。不要原地修改传入的 `logits`，因为调试和复用函数时，原地修改很容易造成隐蔽 bug。

完成这个练习后，你就能把一个“全词表概率分布”裁剪成更适合生成的候选分布。

为了降低难度，top-k 和 top-p 的细节函数已经给出。你只需要补 3 行：temperature 缩放、调用 top-k、调用 top-p。


In [40]:
def _apply_top_k_filter(filtered, top_k):
    """保留 logits 中最大的 top_k 个，其余位置设为 -inf。"""
    if top_k is None or top_k <= 0:
        return filtered
    k = min(int(top_k), filtered.numel())
    threshold = torch.topk(filtered, k).values[-1]
    return filtered.masked_fill(filtered < threshold, float("-inf"))


def _apply_top_p_filter(filtered, top_p):
    """保留累计概率不超过 top_p 的 nucleus token 集合。"""
    if top_p is None or top_p >= 1.0:
        return filtered
    sorted_logits, sorted_indices = torch.sort(filtered, descending=True)
    sorted_probs = torch.softmax(sorted_logits, dim=-1)
    remove_mask = torch.cumsum(sorted_probs, dim=-1) > top_p
    remove_mask[0] = False  # 至少保留概率最高的 token，避免候选集合为空。
    filtered = filtered.clone()
    filtered[sorted_indices[remove_mask]] = float("-inf")
    return filtered


def filter_logits(logits, temperature=1.0, top_k=None, top_p=None):
    """
    logits: 一维 tensor，形状 [vocab_size]
    返回：过滤后的 logits，一维 tensor。
    """
    if logits.ndim != 1:
        raise ValueError("filter_logits 只接收一维 logits")
    if temperature <= 0:
        raise ValueError("temperature 必须大于 0；贪心解码请在外部用 argmax")

    ### START CODE HERE ###
    # TODO 1：复制 logits，并除以 temperature。
    # 提示：不要原地修改输入 logits；使用 logits.clone()。

    # TODO 2：应用 top-k 过滤。

    # TODO 3：应用 top-p 过滤。
    ### END CODE HERE ###

    return filtered

# 玩具例子：第 4 个 token 明显最大。
toy_logits = torch.tensor([0.1, 0.2, 0.3, 4.0, 0.4])
print(filter_logits(toy_logits, temperature=1.0, top_k=2, top_p=None))


tensor([  -inf,   -inf,   -inf, 4.0000, 0.4000])


In [41]:
# 测试练习 5
toy = torch.tensor([0.1, 0.2, 0.3, 4.0, 0.4])
filtered = filter_logits(toy, temperature=1.0, top_k=2, top_p=None)
assert torch.isinf(filtered).sum().item() == 3, "top_k=2 应只保留 2 个 token"
assert filtered[3].isfinite() and filtered[4].isfinite(), "应保留最大两个 logits"
assert torch.allclose(toy, torch.tensor([0.1, 0.2, 0.3, 4.0, 0.4])), "不要原地修改输入 logits"

filtered_p = filter_logits(toy, temperature=1.0, top_k=None, top_p=0.7)
assert filtered_p[3].isfinite(), "top_p 至少应保留概率最高的 token"
assert torch.isinf(filtered_p).sum().item() >= 1, "top_p 应屏蔽一部分低概率 token"
print("练习 5 通过。")


练习 5 通过。


## 8. 自己写一个生成循环

`model.generate` 很方便，但真实发生的是一个循环：输入上下文，预测下一个 token，把 token 接到末尾，再预测下一个 token。这个过程叫自回归生成（autoregressive generation）。

更具体地说，每一步会做以下事情：

1. 把当前 token 序列输入模型。
2. 取最后一个位置的 logits，因为我们只关心下一个 token。
3. 用 greedy、temperature、top-k、top-p 等策略选择 `next_token`。
4. 把 `next_token` 拼接到序列末尾。
5. 如果生成了结束 token，或者达到最大长度，就停止；否则继续下一步。

真实推理系统通常会使用 KV cache：已经计算过的 key/value 不再重复计算，只对新 token 做增量前向传播。本实验为了让逻辑更清楚，每一步都把完整上下文重新喂给模型，因此速度会慢一些，但更适合学习。

还要注意一个常见细节：模型返回的序列包含 prompt 本身。如果想得到 assistant 新生成的内容，需要记住原始输入长度 `original_len`，最后只 decode `generated[0, original_len:]`。

### 练习 6：实现自回归生成

请使用你在练习 5 中写好的 `filter_logits`，完成一个可控的文本生成函数。

这个函数同时支持两种模式：

- `temperature == 0`：使用 `argmax`，即贪心解码。
- `temperature > 0`：先过滤 logits，再用 `torch.multinomial` 从概率分布中抽样。

完成这个练习后，你就拥有了一个迷你版 `generate`，可以直接观察不同采样参数如何改变输出。

本练习只需要补 4 行左右：模型前向、选择下一个 token、拼接 token、保留停止判断。采样细节已经封装在辅助函数里。


In [42]:
def _choose_next_token(last_logits, temperature, top_k, top_p):
    """根据 logits 和采样参数选择下一个 token。"""
    if temperature == 0:
        return torch.argmax(last_logits, dim=-1, keepdim=True)
    filtered_logits = filter_logits(last_logits, temperature=temperature, top_k=top_k, top_p=top_p)
    probs = torch.softmax(filtered_logits, dim=-1)
    return torch.multinomial(probs, num_samples=1)


@torch.no_grad()
def generate_with_controls(
    messages,
    max_new_tokens=80,
    temperature=0.8,
    top_k=40,
    top_p=0.95,
):
    """
    用手写循环生成文本。返回 assistant 新生成的字符串。

    当 temperature == 0 时，使用 argmax 贪心解码；
    当 temperature > 0 时，使用 filter_logits + multinomial 采样。
    """
    inputs = build_chat_inputs(messages, tokenizer, device)
    input_ids = inputs["input_ids"]
    original_len = input_ids.shape[1]
    generated = input_ids.clone()

    ### START CODE HERE ###
    # TODO：完成“预测一个 token -> 拼回上下文 -> 判断是否停止”的循环。
    # 提示 1：model(input_ids=generated).logits[0, -1] 是下一步 logits。
    # 提示 2：用 _choose_next_token(...) 选择 token。
    # 提示 3：用 torch.cat 把 next_token.reshape(1, 1) 接到 generated 后面。
    ### END CODE HERE ###

    new_token_ids = generated[0, original_len:]
    return tokenizer.decode(new_token_ids, skip_special_tokens=True)

messages = [
    {"role": "system", "content": "You are a playful but precise teaching assistant."},
    {"role": "user", "content": "Explain attention using a cooking metaphor in 3 bullets."},
]
print(generate_with_controls(messages, max_new_tokens=80, temperature=0.7, top_k=40, top_p=0.95))


Attention - a cooking metaphor for the ability to focus on a single, crucial ingredient in a dish, while others are ignored.


In [43]:
# 测试练习 6
messages = [{"role": "user", "content": "Write one short sentence about neural networks."}]
text = generate_with_controls(messages, max_new_tokens=20, temperature=0, top_k=None, top_p=None)
assert isinstance(text, str), "生成结果应为字符串"
assert len(text.strip()) > 0, "生成结果不应为空"
assert len(tokenizer.encode(text, add_special_tokens=False)) <= 25, "max_new_tokens 附近不应生成过长文本"
print("练习 6 通过。")


练习 6 通过。


### 小实验：同一个 prompt，不同采样参数

运行下面的格子几次，比较输出的稳定性和发散程度。


In [44]:
prompt_messages = [
    {"role": "system", "content": "You are a creative teaching assistant. Be concise."},
    {"role": "user", "content": "Invent a tiny classroom game for learning self-attention."},
]

settings = [
    {"temperature": 0, "top_k": None, "top_p": None},
    {"temperature": 0.5, "top_k": 20, "top_p": 0.9},
    {"temperature": 1.0, "top_k": 50, "top_p": 0.95},
]

for s in settings:
    print("\nSETTINGS:", s)
    print(generate_with_controls(prompt_messages, max_new_tokens=70, **s))



SETTINGS: {'temperature': 0, 'top_k': None, 'top_p': None}
Imagine a tiny classroom game where students learn to focus their attention and manage their thoughts. Here's a simple yet effective game:

**Game:**

**Objective:** Students will learn to focus their attention and manage their thoughts in a tiny classroom.

**Game:**

**Objective:** Students will learn to focus their attention and manage

SETTINGS: {'temperature': 0.5, 'top_k': 20, 'top_p': 0.9}
Imagine a tiny classroom game where students learn to focus their attention to achieve a specific goal. Here's a simple game to get you started:

**Game:**

**Objective:** Students will learn to focus their attention to achieve a specific goal, such as completing a task or solving a puzzle.

**Game:**

**Objective

SETTINGS: {'temperature': 1.0, 'top_k': 50, 'top_p': 0.95}
A tiny classroom game for learning self-attention.

1. Classrooms: Each classroom consists of four 15x15 room windows on either side of a small, rectangular platfor

## 9. 模型更喜欢哪个续写？

给定一个开头和两个候选续写，语言模型可以用负对数似然（NLL）衡量哪个续写更符合它学到的分布。NLL 越低，模型越“喜欢”。

如果 continuation 由 token $x_1, x_2, \ldots, x_T$ 组成，模型给它的平均负对数似然可以理解为：

$$
\mathrm{NLL} = -\frac{1}{T}\sum_{t=1}^T \log p(x_t \mid \mathrm{prompt}, x_{<t})
$$

困惑度（perplexity, PPL）是 NLL 的指数形式：

$$
\mathrm{PPL}=\exp(\mathrm{NLL})
$$

直觉上，PPL 可以粗略理解为“模型在每一步平均有多少个差不多的选择”。PPL 越低，说明模型越容易预测这段文本。但要小心：低 PPL 不等于事实正确，只说明这段文字更符合模型训练中学到的统计模式。

在 Hugging Face 的 causal language model 中，如果传入 `labels`，模型会自动计算 next-token loss。`labels=-100` 的位置会被忽略，不参与损失计算。我们正好可以利用这一点：把 prompt 部分 label 设成 `-100`，只计算 continuation 部分的 loss。

### 练习 7：实现续写打分

这个练习会把 prompt 部分的 label 设成 `-100`，让损失只计算 continuation 部分。

实现时请注意：prompt 和 continuation 拼接后再 tokenize，和分别 tokenize 后拼接 token id 不一定完全一样，因为 tokenizer 的切分可能受边界空格影响。本实验为了简单，会分别计算 prompt token 数，再对完整文本打分；写 prompt 时最好让 prompt 以空格或明显标点结尾，减少边界影响。

完成这个练习后，你可以让模型比较两个续写：哪个更像自然语言，哪个更像 Python 代码，哪个更像某种写作风格。

本练习只需要补 5 行：编码 prompt、编码完整文本、构造 labels、把 prompt 部分设为 `-100`、调用模型计算 loss。


In [45]:
def _text_to_input_ids(text):
    """把字符串编码成形状 [1, seq_len] 的 LongTensor。"""
    ids = tokenizer.encode(text, add_special_tokens=False)
    return torch.tensor([ids], dtype=torch.long, device=device)


def _loss_to_nll_ppl(loss):
    """把模型 loss 转成 Python float 形式的 NLL 和 PPL。"""
    nll = float(loss.item())
    ppl = float(math.exp(min(nll, 20)))
    return nll, ppl


@torch.no_grad()
def score_continuation(prompt, continuation):
    """
    返回 continuation 的平均 NLL 和困惑度 perplexity。

    为了减少 BPE 边界带来的干扰，建议 prompt 以空格或明显标点结尾。
    """
    ### START CODE HERE ###
    # TODO 1：先单独编码 prompt，用来知道 prompt 占多少 token。

    # TODO 2：编码 prompt + continuation，作为模型输入。

    # TODO 3：复制 input_ids 得到 labels，然后屏蔽 prompt 部分。

    # TODO 4：把 input_ids 和 labels 喂给模型，得到带 loss 的输出。
    ### END CODE HERE ###

    return _loss_to_nll_ppl(outputs.loss)

prompt = "The most important idea in self-attention is "
choices = [
    "each token can mix information from other tokens.",
    "the model sorts images by their file names.",
]
for c in choices:
    nll, ppl = score_continuation(prompt, c)
    print(f"NLL={nll:.3f} PPL={ppl:.1f} | {c}")


NLL=4.573 PPL=96.8 | each token can mix information from other tokens.
NLL=5.413 PPL=224.3 | the model sorts images by their file names.


In [46]:
# 测试练习 7
nll, ppl = score_continuation("Deep learning is ", "a branch of machine learning.")
assert isinstance(nll, float) and isinstance(ppl, float), "应返回两个 float"
assert nll > 0 and ppl > 1, "NLL/PPL 应为正数"
print("练习 7 通过。")


练习 7 通过。


### 小实验：让模型做裁判

尝试自己写三组 `prompt + continuation`。例子：

1. 常识判断：`Water freezes at ` + `zero degrees Celsius.` / `one hundred degrees Celsius.`
2. 代码判断：`def add(a, b): return ` + `a + b` / `a - b`
3. 风格判断：`In Shakespeare style, the moon is ` + 两个不同风格的续写

问题：模型偏好的续写一定正确吗？如果不一定，偏好来自哪里？


In [47]:
my_prompt = "In Python, a function starts with the keyword "
my_choices = [
    "def.",
    "banana.",
]

for c in my_choices:
    nll, ppl = score_continuation(my_prompt, c)
    print(f"NLL={nll:.3f} PPL={ppl:.1f} | {c}")


NLL=1.507 PPL=4.5 | def.
NLL=1.611 PPL=5.0 | banana.


## 10. 课堂挑战：Prompt 设计与故障分析

下面不是自动评分题，而是让你把模型当作实验对象。每个挑战都建议记录 prompt、参数和输出。做这些挑战时，重点不是得到“最好看”的回答，而是解释模型行为为什么变化。

一个好的实验记录至少包括：

- 你给模型的 system prompt 和 user prompt。
- 生成参数，例如 `temperature`、`top_k`、`top_p`、`max_new_tokens`。
- 模型输出中的关键片段。
- 你的分析：变化来自 prompt、采样参数、模型能力，还是上下文长度和语言差异？

### 挑战 A：温度盲盒

固定一个 prompt，分别用 `temperature=0`、`0.5`、`1.0` 生成三次。比较：哪一个最稳定？哪一个最有创意？哪一个开始跑题？

分析提示：`temperature=0` 使用贪心解码，因此同样输入通常得到同样输出；高 temperature 会放大随机性，但不保证更聪明，只是更愿意探索低概率 token。

### 挑战 B：角色约束

让模型扮演“严谨的助教”，要求它只用三句话解释 attention。再把 system prompt 改成“喜欢讲故事的助教”。比较输出是否真的变了。

分析提示：system prompt 是软约束，不是程序里的硬规则。小模型可能部分遵守，也可能忽略。你可以观察它是否遵守句数、语气和内容边界。

### 挑战 C：模型侦探

找一个模型明显答错的问题。不要只写“它错了”，请分析可能原因：模型太小、训练数据偏差、prompt 不清晰、还是采样太发散？

分析提示：语言模型的训练目标是预测文本，不是维护一个可靠数据库。它可能生成听起来合理但事实错误的句子。降低 temperature 有时能减少发散，但不能保证事实正确。

### 挑战 D：反事实 token

把同一句 prompt 中的一个词替换掉，例如 `easy` 改成 `difficult`，用 `next_token_candidates` 比较下一个 token 的概率变化。

分析提示：这能展示语言模型对上下文的敏感性。你不必只看最终生成文本，也可以看 top token 概率如何移动；这比只看一段随机输出更容易分析。

挑战题不要求长代码。建议每个挑战先写 3 到 5 行：定义 prompt、调用一个函数、打印结果，然后把观察写进实验报告。


In [48]:
# 在这里完成挑战 A/B/C/D。建议每次实验都保留 prompt 和参数，方便复盘。

### START CODE HERE ###
# TODO：把 pass 替换成 3 到 5 行自己的实验代码。
# 提示：可以先构造 messages，再调用 generate_with_controls(...)。
# 提示：也可以调用 next_token_candidates(...) 比较两个 prompt 的下一个 token。
### END CODE HERE ###


0 Decoder self-attention is a fundamental component of attention-based neural networks (NNs) that aims to capture the context of an input image or text. The self-attention mechanism is designed to learn the context of an input image or text by
0.5 The decoder needs a causal mask because it is designed to learn the underlying causal relationships between the input and output signals. The decoder is trained to recognize the underlying causal relationships between the input and output signals, and to learn the causal relationships between the input and
1.0 Dcoder Self-Attention requires a causal mask due to its dependence on the training data. In the context of decoder training, a causal mask provides a crucial layer of prior knowledge that serves as a baseline, allowing the decoder to learn the optimal parameters
This task is easy because [{'rank': 1, 'token_id': 57, 'token': 'I', 'prob': 0.15532125532627106}, {'rank': 2, 'token_id': 1348, 'token': 'This', 'prob': 0.14813904464244843}, {

## 11. 总结问题

请在实验报告里回答：

1. tokenizer 把中文和英文切分成 token 时，有什么差异？
2. causal mask 为什么必须屏蔽未来 token？如果不屏蔽，训练和生成会发生什么问题？
3. temperature、top-k、top-p 分别如何影响生成结果？
4. 你观察到的小模型局限是什么？这些局限是 Transformer 架构本身导致的吗？
5. 如果要把这个实验扩展成一个更强的中文本地助手，你会优先换模型、换 prompt，还是做微调？为什么？
